In [ ]:
import importlib.util
import subprocess
import sys

if any(importlib.util.find_spec(name) is None for name in ("numpy", "matplotlib")):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "numpy>=1.24,<3", "matplotlib>=3.7,<4",
    ])

In [ ]:
"""Deterministic clean and noisy XOR datasets."""

from dataclasses import dataclass

import numpy as np
from numpy.typing import NDArray


FloatArray = NDArray[np.float64]
IntArray = NDArray[np.int64]


@dataclass(frozen=True, eq=False)
class NoisyXORData:
    X: FloatArray
    y: IntArray
    axis_crossing_count: int
    axis_crossing_rate: float


@dataclass(frozen=True, eq=False)
class DataSplit:
    X_train: FloatArray
    X_test: FloatArray
    y_train: IntArray
    y_test: IntArray


def clean_xor() -> tuple[FloatArray, IntArray]:
    """Return the four XOR corners and labels in {-1, +1}."""

    X = np.array([[-1, -1], [-1, 1], [1, -1], [1, 1]], dtype=np.float64)
    y = np.array([-1, 1, 1, -1], dtype=np.int64)
    return X, y


def make_noisy_xor_clusters(
    samples_per_corner: int = 50,
    noise_std: float = 0.2,
    *,
    seed: int = 0,
) -> NoisyXORData:
    """Sample balanced Gaussian clusters around the four XOR corners."""

    if samples_per_corner < 1:
        raise ValueError("samples_per_corner must be positive")
    if noise_std < 0:
        raise ValueError("noise_std must be non-negative")

    corners, corner_labels = clean_xor()
    rng = np.random.default_rng(seed)
    samples = []
    labels = []
    for corner, label in zip(corners, corner_labels):
        samples.append(
            rng.normal(
                loc=corner,
                scale=noise_std,
                size=(samples_per_corner, corner.shape[0]),
            )
        )
        labels.append(np.full(samples_per_corner, label, dtype=np.int64))

    X = np.vstack(samples).astype(np.float64)
    y = np.concatenate(labels)
    products = X[:, 0] * X[:, 1]
    ideal_labels = np.where(products <= 0.0, 1, -1)
    crossing_count = int(np.count_nonzero(ideal_labels != y))

    return NoisyXORData(
        X=X,
        y=y,
        axis_crossing_count=crossing_count,
        axis_crossing_rate=crossing_count / y.size,
    )


def stratified_split(
    X: NDArray[np.generic],
    y: NDArray[np.generic],
    *,
    test_fraction: float = 0.25,
    seed: int = 0,
) -> DataSplit:
    """Split each class independently so both subsets preserve the classes."""

    features = np.asarray(X, dtype=np.float64)
    labels = np.asarray(y, dtype=np.int64)
    if features.ndim != 2 or labels.ndim != 1 or features.shape[0] != labels.shape[0]:
        raise ValueError("X and y must contain the same number of samples")
    if not 0.0 < test_fraction < 1.0:
        raise ValueError("test_fraction must be strictly between 0 and 1")

    classes = np.unique(labels)
    if classes.size < 2:
        raise ValueError("at least two classes are required")

    rng = np.random.default_rng(seed)
    train_parts = []
    test_parts = []
    for label in classes:
        indices = np.flatnonzero(labels == label)
        if indices.size < 2:
            raise ValueError("each class needs at least two samples")
        shuffled = rng.permutation(indices)
        test_count = int(round(indices.size * test_fraction))
        test_count = min(max(test_count, 1), indices.size - 1)
        test_parts.append(shuffled[:test_count])
        train_parts.append(shuffled[test_count:])

    train_indices = rng.permutation(np.concatenate(train_parts))
    test_indices = rng.permutation(np.concatenate(test_parts))
    return DataSplit(
        X_train=features[train_indices].copy(),
        X_test=features[test_indices].copy(),
        y_train=labels[train_indices].copy(),
        y_test=labels[test_indices].copy(),
    )

In [ ]:
"""Feature maps used to study linear separability of XOR."""

from dataclasses import dataclass
from typing import Callable, Dict, Tuple

import numpy as np
from numpy.typing import NDArray


FloatArray = NDArray[np.float64]


@dataclass(frozen=True)
class FeatureMap:
    """A deterministic mapping from two inputs to a labelled feature space."""

    feature_names: Tuple[str, ...]
    transform: Callable[[FloatArray], FloatArray]


def _raw(X: FloatArray) -> FloatArray:
    return np.column_stack((np.ones(X.shape[0]), X))


def _square(X: FloatArray) -> FloatArray:
    return np.column_stack((np.ones(X.shape[0]), X, X[:, 0] ** 2, X[:, 1] ** 2))


def _interaction(X: FloatArray) -> FloatArray:
    return np.column_stack((np.ones(X.shape[0]), X, X[:, 0] * X[:, 1]))


def _quadratic(X: FloatArray) -> FloatArray:
    return np.column_stack(
        (
            np.ones(X.shape[0]),
            X,
            X[:, 0] ** 2,
            X[:, 1] ** 2,
            X[:, 0] * X[:, 1],
        )
    )


FEATURE_MAPS: Dict[str, FeatureMap] = {
    "raw": FeatureMap(("1", "x1", "x2"), _raw),
    "square": FeatureMap(("1", "x1", "x2", "x1^2", "x2^2"), _square),
    "interaction": FeatureMap(("1", "x1", "x2", "x1*x2"), _interaction),
    "quadratic": FeatureMap(
        ("1", "x1", "x2", "x1^2", "x2^2", "x1*x2"), _quadratic
    ),
}


def transform_features(X: NDArray[np.generic], name: str) -> FloatArray:
    """Apply a named feature map and include an explicit intercept column."""

    values = np.asarray(X, dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != 2:
        raise ValueError("X must have shape (n_samples, 2)")

    try:
        feature_map = FEATURE_MAPS[name]
    except KeyError as error:
        available = ", ".join(FEATURE_MAPS)
        raise ValueError(f"Unknown feature map {name!r}; choose from {available}") from error

    return np.asarray(feature_map.transform(values), dtype=np.float64)

In [ ]:
"""A small, deterministic implementation of the binary Perceptron."""

from dataclasses import dataclass
from typing import Tuple

import numpy as np
from numpy.typing import NDArray


FloatArray = NDArray[np.float64]
IntArray = NDArray[np.int64]


@dataclass(frozen=True)
class PerceptronResult:
    """Training result and the evidence needed to discuss convergence."""

    weights: FloatArray
    converged: bool
    epochs_run: int
    updates: int
    mistakes_per_epoch: Tuple[int, ...]
    empirical_error: float


def _as_feature_matrix(Phi: NDArray[np.generic]) -> FloatArray:
    values = np.asarray(Phi, dtype=np.float64)
    if values.ndim != 2 or values.shape[0] == 0 or values.shape[1] == 0:
        raise ValueError("Phi must be a non-empty two-dimensional feature matrix")
    return values


def predict_scores(Phi: NDArray[np.generic], weights: NDArray[np.generic]) -> FloatArray:
    """Return the linear score ``Phi @ weights`` for every sample."""

    features = _as_feature_matrix(Phi)
    coefficients = np.asarray(weights, dtype=np.float64)
    if coefficients.ndim != 1 or features.shape[1] != coefficients.shape[0]:
        raise ValueError("weights must be one-dimensional and match the feature width")
    return features @ coefficients


def predict_labels(Phi: NDArray[np.generic], weights: NDArray[np.generic]) -> IntArray:
    """Predict -1 or +1; a score of exactly zero is assigned to +1."""

    scores = predict_scores(Phi, weights)
    return np.where(scores >= 0.0, 1, -1).astype(np.int64)


def empirical_error(y_true: NDArray[np.generic], y_pred: NDArray[np.generic]) -> float:
    """Return the fraction of labels predicted incorrectly."""

    expected = np.asarray(y_true)
    predicted = np.asarray(y_pred)
    if expected.ndim != 1 or predicted.ndim != 1 or expected.shape != predicted.shape:
        raise ValueError("true and predicted labels must be equally sized one-dimensional arrays")
    if expected.size == 0:
        raise ValueError("labels must not be empty")
    return float(np.mean(expected != predicted))


def fit_perceptron(
    Phi: NDArray[np.generic],
    y: NDArray[np.generic],
    *,
    learning_rate: float = 1.0,
    max_epochs: int = 100,
    seed: int = 0,
) -> PerceptronResult:
    """Fit a binary Perceptron using a reproducible shuffle each epoch."""

    features = _as_feature_matrix(Phi)
    labels = np.asarray(y, dtype=np.int64)
    if labels.ndim != 1 or labels.shape[0] != features.shape[0]:
        raise ValueError("labels must be one-dimensional and match the sample count")
    if not np.all(np.isin(labels, (-1, 1))):
        raise ValueError("labels must contain only -1 and +1")
    if learning_rate <= 0:
        raise ValueError("learning_rate must be positive")
    if max_epochs < 1:
        raise ValueError("max_epochs must be at least 1")

    rng = np.random.default_rng(seed)
    weights = np.zeros(features.shape[1], dtype=np.float64)
    mistakes_history = []
    updates = 0
    converged = False

    for _ in range(max_epochs):
        mistakes = 0
        for index in rng.permutation(features.shape[0]):
            score = float(features[index] @ weights)
            prediction = 1 if score >= 0.0 else -1
            if prediction != labels[index]:
                weights += learning_rate * labels[index] * features[index]
                updates += 1
                mistakes += 1
        mistakes_history.append(mistakes)
        if mistakes == 0:
            converged = True
            break

    predictions = predict_labels(features, weights)
    return PerceptronResult(
        weights=weights.copy(),
        converged=converged,
        epochs_run=len(mistakes_history),
        updates=updates,
        mistakes_per_epoch=tuple(mistakes_history),
        empirical_error=empirical_error(labels, predictions),
    )

In [ ]:
"""Pure orchestration for the clean and noisy XOR experiments."""

from dataclasses import dataclass
from typing import Optional, Sequence, Tuple

import numpy as np
from numpy.typing import NDArray



DEFAULT_TRAINING_SEEDS = tuple(range(20))


@dataclass(frozen=True)
class ExperimentResult:
    feature_map: str
    seeds: Tuple[int, ...]
    convergence_rate: float
    mean_updates: float
    std_updates: float
    mean_epochs: float
    std_epochs: float
    mean_train_error: float
    std_train_error: float
    mean_test_error: Optional[float]
    std_test_error: Optional[float]
    weights: Tuple[Tuple[float, ...], ...]
    mistakes_per_epoch: Tuple[Tuple[int, ...], ...]
    updates: Tuple[int, ...]
    epochs: Tuple[int, ...]


@dataclass(frozen=True)
class NoisyExperiment:
    seed: int
    samples_per_corner: int
    noise_std: float
    test_fraction: float
    train_samples: int
    test_samples: int
    axis_crossing_count: int
    axis_crossing_rate: float
    results: Tuple[ExperimentResult, ...]


def _mean_std(values: Sequence[float]) -> tuple[float, float]:
    array = np.asarray(values, dtype=np.float64)
    return float(np.mean(array)), float(np.std(array))


def _run_repeated(
    feature_map: str,
    X_train: NDArray[np.generic],
    y_train: NDArray[np.generic],
    *,
    seeds: Sequence[int],
    max_epochs: int,
    test_data: Optional[tuple[NDArray[np.generic], NDArray[np.generic]]] = None,
) -> ExperimentResult:
    seed_tuple = tuple(int(seed) for seed in seeds)
    if not seed_tuple:
        raise ValueError("at least one training seed is required")

    train_features = transform_features(X_train, feature_map)
    test_features = None
    test_labels = None
    if test_data is not None:
        X_test, y_test = test_data
        test_features = transform_features(X_test, feature_map)
        test_labels = np.asarray(y_test, dtype=np.int64)

    runs = [
        fit_perceptron(
            train_features,
            y_train,
            learning_rate=1.0,
            max_epochs=max_epochs,
            seed=seed,
        )
        for seed in seed_tuple
    ]
    train_errors = [run.empirical_error for run in runs]
    test_errors = None
    if test_features is not None and test_labels is not None:
        test_errors = [
            empirical_error(test_labels, predict_labels(test_features, run.weights))
            for run in runs
        ]

    mean_updates, std_updates = _mean_std([run.updates for run in runs])
    mean_epochs, std_epochs = _mean_std([run.epochs_run for run in runs])
    mean_train_error, std_train_error = _mean_std(train_errors)
    if test_errors is None:
        mean_test_error = None
        std_test_error = None
    else:
        mean_test_error, std_test_error = _mean_std(test_errors)

    return ExperimentResult(
        feature_map=feature_map,
        seeds=seed_tuple,
        convergence_rate=float(np.mean([run.converged for run in runs])),
        mean_updates=mean_updates,
        std_updates=std_updates,
        mean_epochs=mean_epochs,
        std_epochs=std_epochs,
        mean_train_error=mean_train_error,
        std_train_error=std_train_error,
        mean_test_error=mean_test_error,
        std_test_error=std_test_error,
        weights=tuple(tuple(float(value) for value in run.weights) for run in runs),
        mistakes_per_epoch=tuple(run.mistakes_per_epoch for run in runs),
        updates=tuple(run.updates for run in runs),
        epochs=tuple(run.epochs_run for run in runs),
    )


def run_clean_experiment(
    *,
    seeds: Sequence[int] = DEFAULT_TRAINING_SEEDS,
    max_epochs: int = 100,
) -> Tuple[ExperimentResult, ...]:
    """Run every feature map on the exact four-point XOR dataset."""

    X, y = clean_xor()
    return tuple(
        _run_repeated(name, X, y, seeds=seeds, max_epochs=max_epochs)
        for name in FEATURE_MAPS
    )


def run_noisy_experiment(
    *,
    seed: int = 13,
    samples_per_corner: int = 100,
    noise_std: float = 0.35,
    test_fraction: float = 0.25,
    training_seeds: Sequence[int] = DEFAULT_TRAINING_SEEDS,
    max_epochs: int = 200,
) -> NoisyExperiment:
    """Compare raw and interaction features on one reproducible noisy split."""

    data = make_noisy_xor_clusters(
        samples_per_corner=samples_per_corner,
        noise_std=noise_std,
        seed=seed,
    )
    split = stratified_split(
        data.X,
        data.y,
        test_fraction=test_fraction,
        seed=seed,
    )
    results = tuple(
        _run_repeated(
            name,
            split.X_train,
            split.y_train,
            test_data=(split.X_test, split.y_test),
            seeds=training_seeds,
            max_epochs=max_epochs,
        )
        for name in ("raw", "interaction")
    )
    return NoisyExperiment(
        seed=seed,
        samples_per_corner=samples_per_corner,
        noise_std=noise_std,
        test_fraction=test_fraction,
        train_samples=split.y_train.size,
        test_samples=split.y_test.size,
        axis_crossing_count=data.axis_crossing_count,
        axis_crossing_rate=data.axis_crossing_rate,
        results=results,
    )

In [ ]:
"""Generate deterministic tables and figures for the XOR study."""

import csv
import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable, Tuple

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap



NEGATIVE_COLOUR = "#355C7D"
POSITIVE_COLOUR = "#C06C84"
MAP_COLOURS = ("#6C5B7B", "#4C956C", "#F2A65A", "#D1495B")


@dataclass(frozen=True)
class OutputManifest:
    clean_csv: Path
    noisy_csv: Path
    summary_json: Path
    xor_input_space: Path
    feature_map_comparison: Path
    interaction_feature_separation: Path
    mistakes_by_epoch: Path
    noisy_decision_regions: Path

    @property
    def all_paths(self) -> Tuple[Path, ...]:
        return (
            self.clean_csv,
            self.noisy_csv,
            self.summary_json,
            self.xor_input_space,
            self.feature_map_comparison,
            self.interaction_feature_separation,
            self.mistakes_by_epoch,
            self.noisy_decision_regions,
        )


def _result_row(result: ExperimentResult) -> dict:
    return {
        "feature_map": result.feature_map,
        "runs": len(result.seeds),
        "convergence_rate": result.convergence_rate,
        "mean_updates": result.mean_updates,
        "std_updates": result.std_updates,
        "mean_epochs": result.mean_epochs,
        "std_epochs": result.std_epochs,
        "mean_train_error": result.mean_train_error,
        "std_train_error": result.std_train_error,
        "mean_test_error": result.mean_test_error,
        "std_test_error": result.std_test_error,
    }


def _write_results_csv(path: Path, results: Iterable[ExperimentResult]) -> None:
    rows = [_result_row(result) for result in results]
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=tuple(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def _save_figure(figure: plt.Figure, path: Path) -> None:
    figure.tight_layout()
    figure.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(figure)


def _plot_input_space(path: Path) -> None:
    X, y = clean_xor()
    figure, axis = plt.subplots(figsize=(5.6, 5.0))
    for label, colour, name in (
        (-1, NEGATIVE_COLOUR, "Class -1"),
        (1, POSITIVE_COLOUR, "Class +1"),
    ):
        mask = y == label
        axis.scatter(
            X[mask, 0],
            X[mask, 1],
            s=130,
            c=colour,
            edgecolors="white",
            linewidths=1.5,
            label=name,
        )
    axis.axhline(0, color="#999999", linewidth=0.8)
    axis.axvline(0, color="#999999", linewidth=0.8)
    axis.set(xlabel="$x_1$", ylabel="$x_2$", title="XOR in the original input space")
    axis.set_xticks((-1, 0, 1))
    axis.set_yticks((-1, 0, 1))
    axis.set_aspect("equal")
    axis.legend()
    _save_figure(figure, path)


def _plot_feature_comparison(
    path: Path, results: Tuple[ExperimentResult, ...]
) -> None:
    names = [result.feature_map for result in results]
    x = np.arange(len(names))
    figure, (convergence_axis, error_axis) = plt.subplots(1, 2, figsize=(10.5, 4.2))
    convergence_axis.bar(
        x,
        [result.convergence_rate for result in results],
        color=MAP_COLOURS,
    )
    convergence_axis.set(
        xticks=x,
        xticklabels=names,
        ylim=(0, 1.08),
        ylabel="Convergence rate",
        title=f"{len(results[0].seeds)} shuffled training orders",
    )
    error_axis.bar(
        x,
        [result.mean_train_error for result in results],
        color=MAP_COLOURS,
    )
    error_axis.set(
        xticks=x,
        xticklabels=names,
        ylim=(0, 0.55),
        ylabel="Mean empirical error",
        title="Final error on four XOR points",
    )
    for axis in (convergence_axis, error_axis):
        axis.tick_params(axis="x", rotation=25)
        axis.grid(axis="y", alpha=0.25)
    figure.suptitle("Feature map determines whether a linear model can represent XOR")
    _save_figure(figure, path)


def _plot_interaction_separation(path: Path) -> None:
    X, y = clean_xor()
    interaction = transform_features(X, "interaction")[:, -1]
    offsets = np.array([-0.06, 0.06, -0.06, 0.06])
    figure, axis = plt.subplots(figsize=(8.0, 3.2))
    axis.axvspan(-1.5, 0.0, color="#F5DDE5", alpha=0.75)
    axis.axvspan(0.0, 1.5, color="#DCEAF3", alpha=0.75)
    axis.axvline(0.0, color="#263238", linestyle="--", linewidth=1.4)
    for label, colour, name in (
        (-1, NEGATIVE_COLOUR, "Class -1"),
        (1, POSITIVE_COLOUR, "Class +1"),
    ):
        mask = y == label
        axis.scatter(
            interaction[mask],
            offsets[mask],
            s=120,
            c=colour,
            edgecolors="white",
            linewidths=1.5,
            label=name,
            zorder=3,
        )
    axis.text(-0.75, 0.14, "predict +1", ha="center", color=POSITIVE_COLOUR)
    axis.text(0.75, 0.14, "predict -1", ha="center", color=NEGATIVE_COLOUR)
    axis.set(
        xlim=(-1.5, 1.5),
        ylim=(-0.16, 0.22),
        xticks=(-1, 0, 1),
        yticks=[],
        xlabel=FEATURE_MAPS["interaction"].feature_names[-1],
        title="One-dimensional separation after adding the interaction feature",
    )
    axis.legend(loc="lower center", ncol=2)
    _save_figure(figure, path)


def _plot_mistakes(path: Path, results: Tuple[ExperimentResult, ...]) -> None:
    figure, axis = plt.subplots(figsize=(8.2, 4.8))
    for result, colour in zip(results, MAP_COLOURS):
        history = result.mistakes_per_epoch[0]
        axis.plot(
            np.arange(1, len(history) + 1),
            history,
            label=result.feature_map,
            color=colour,
            linewidth=2,
        )
    axis.set(
        xlabel="Epoch",
        ylabel="Number of updates",
        title="Perceptron mistakes: interaction features converge; raw features cycle",
    )
    axis.set_ylim(bottom=0)
    axis.grid(alpha=0.25)
    axis.legend()
    _save_figure(figure, path)


def _plot_noisy_regions(path: Path, experiment: NoisyExperiment) -> None:
    data = make_noisy_xor_clusters(
        experiment.samples_per_corner,
        experiment.noise_std,
        seed=experiment.seed,
    )
    grid_values = np.linspace(-2.1, 2.1, 220)
    grid_x1, grid_x2 = np.meshgrid(grid_values, grid_values)
    grid = np.column_stack((grid_x1.ravel(), grid_x2.ravel()))
    background = ListedColormap(("#DCEAF3", "#F5DDE5"))
    figure, axes = plt.subplots(1, 2, figsize=(10.5, 4.8), sharex=True, sharey=True)
    for axis, result in zip(axes, experiment.results):
        weights = np.asarray(result.weights[0], dtype=np.float64)
        predictions = predict_labels(transform_features(grid, result.feature_map), weights)
        axis.contourf(
            grid_x1,
            grid_x2,
            predictions.reshape(grid_x1.shape),
            levels=(-1.5, 0, 1.5),
            cmap=background,
            alpha=0.8,
        )
        for label, colour in ((-1, NEGATIVE_COLOUR), (1, POSITIVE_COLOUR)):
            mask = data.y == label
            axis.scatter(
                data.X[mask, 0],
                data.X[mask, 1],
                c=colour,
                s=12,
                alpha=0.65,
                edgecolors="none",
                label=f"Class {label:+d}",
            )
        axis.set(
            xlabel="$x_1$",
            ylabel="$x_2$",
            title=(
                f"{result.feature_map}: region seed {result.seeds[0]}; "
                f"mean test error {result.mean_test_error:.3f}"
            ),
        )
        axis.legend(loc="upper right", fontsize=8)
    figure.suptitle(
        "Noisy XOR decision regions (colours show predictions; points show labels)"
    )
    _save_figure(figure, path)


def generate_all_outputs(output_root: Path, seed: int = 42) -> OutputManifest:
    """Generate tables and figures under ``output_root``."""

    root = Path(output_root)
    results_dir = root / "results"
    figures_dir = root / "figures"
    results_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    training_seeds = DEFAULT_TRAINING_SEEDS
    clean_results = run_clean_experiment(seeds=training_seeds)
    noisy_experiment = run_noisy_experiment(seed=seed, training_seeds=training_seeds)
    manifest = OutputManifest(
        clean_csv=results_dir / "clean_results.csv",
        noisy_csv=results_dir / "noisy_results.csv",
        summary_json=results_dir / "experiment_summary.json",
        xor_input_space=figures_dir / "xor_input_space.png",
        feature_map_comparison=figures_dir / "feature_map_comparison.png",
        interaction_feature_separation=figures_dir
        / "interaction_feature_separation.png",
        mistakes_by_epoch=figures_dir / "mistakes_by_epoch.png",
        noisy_decision_regions=figures_dir / "noisy_decision_regions.png",
    )

    _write_results_csv(manifest.clean_csv, clean_results)
    _write_results_csv(manifest.noisy_csv, noisy_experiment.results)
    summary = {
        "seed": seed,
        "clean": {
            "training_seeds": list(training_seeds),
            "results": [asdict(result) for result in clean_results],
        },
        "noisy": asdict(noisy_experiment),
    }
    manifest.summary_json.write_text(
        json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )

    _plot_input_space(manifest.xor_input_space)
    _plot_feature_comparison(manifest.feature_map_comparison, clean_results)
    _plot_interaction_separation(manifest.interaction_feature_separation)
    _plot_mistakes(manifest.mistakes_by_epoch, clean_results)
    _plot_noisy_regions(manifest.noisy_decision_regions, noisy_experiment)
    return manifest

In [ ]:
from pathlib import Path
from IPython.display import Image, display

ROOT = Path.cwd()

np.set_printoptions(suppress=True, precision=3)
manifest = generate_all_outputs(ROOT, seed=42)

In [ ]:
X, y = clean_xor()
print("X =")
print(X)
print("y =", y)
display(Image(filename=manifest.xor_input_space))

for name in FEATURE_MAPS:
    print(f"{name:11s} ->")
    print(transform_features(X, name))

In [ ]:
interaction_features = transform_features(X, "interaction")
separator = np.array([0.0, 0.0, 0.0, -1.0])
assert np.array_equal(predict_labels(interaction_features, separator), y)
print("scores =", interaction_features @ separator)
display(Image(filename=manifest.interaction_feature_separation))

In [ ]:
clean_results = run_clean_experiment(seeds=DEFAULT_TRAINING_SEEDS)
print("map          convergence  mean updates  mean epochs  train error")
for result in clean_results:
    print(
        f"{result.feature_map:11s} {result.convergence_rate:11.2f} "
        f"{result.mean_updates:13.2f} {result.mean_epochs:12.2f} "
        f"{result.mean_train_error:11.3f}"
    )
display(Image(filename=manifest.feature_map_comparison))
display(Image(filename=manifest.mistakes_by_epoch))

In [ ]:
data = make_noisy_xor_clusters(samples_per_corner=100, noise_std=0.35, seed=42)
split = stratified_split(data.X, data.y, test_fraction=0.25, seed=42)

def count_axis_crossings(X, y):
    ideal_labels = np.where(X[:, 0] * X[:, 1] <= 0.0, 1, -1)
    return int(np.count_nonzero(ideal_labels != y))

print({
    "train_samples": split.y_train.size,
    "test_samples": split.y_test.size,
    "train_axis_crossings": count_axis_crossings(split.X_train, split.y_train),
    "test_axis_crossings": count_axis_crossings(split.X_test, split.y_test),
})

noisy = run_noisy_experiment(seed=42, training_seeds=DEFAULT_TRAINING_SEEDS)
print("map          convergence  train error (mean, SD)  test error (mean, SD)")
for result in noisy.results:
    print(
        f"{result.feature_map:11s} {result.convergence_rate:11.2f} "
        f"{result.mean_train_error:10.4f} {result.std_train_error:8.4f} "
        f"{result.mean_test_error:10.4f} {result.std_test_error:8.4f}"
    )
display(Image(filename=manifest.noisy_decision_regions))